In [0]:
# ============================================================
# CMD 01 — LOAD SOURCE DATA FOR NOTEBOOK 02
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Load source table
# ------------------------------------------------------------

db_ml = (
    spark.table("ml_perovskites.db_ml")
    .toPandas()
)

print("db_ml loaded successfully.")
print("Shape:", db_ml.shape)

# ------------------------------------------------------------
# Define actual column names
# ------------------------------------------------------------

COMPOUND_COL = "Compound"

BANDGAP_TARGET = "bandgap_energy"

HYDROGEN_TARGET = "hydrogen_production_rate"

DOI_COL = "doi_reference"

# ------------------------------------------------------------
# Validate required columns
# ------------------------------------------------------------

required_columns = [
    COMPOUND_COL,
    BANDGAP_TARGET,
    HYDROGEN_TARGET,
    DOI_COL
]

missing_columns = [
    c for c in required_columns
    if c not in db_ml.columns
]

if missing_columns:
    raise RuntimeError(
        f"Required columns missing: {missing_columns}"
    )

print("\nRequired columns found.")

# ------------------------------------------------------------
# Target summary
# ------------------------------------------------------------

print("\nDataset:")
print(f"Rows:    {db_ml.shape[0]}")
print(f"Columns: {db_ml.shape[1]}")

print("\nTargets:")
print(f"Bandgap:  {BANDGAP_TARGET}")
print(f"Hydrogen: {HYDROGEN_TARGET}")

print("\nMissing target values:")

print(
    db_ml[
        [BANDGAP_TARGET, HYDROGEN_TARGET]
    ].isna().sum()
)

print("\nTarget data types:")

print(
    db_ml[
        [BANDGAP_TARGET, HYDROGEN_TARGET]
    ].dtypes
)

In [0]:
# ============================================================
# CMD 02 — DEFINE TARGETS AND IDENTIFIERS
# ============================================================

COMPOUND_COL = "Compound"

BANDGAP_TARGET = "bandgap_energy"

HYDROGEN_TARGET = "hydrogen_production_rate"

DOI_COL = "doi_reference"

# ------------------------------------------------------------
# Validate required columns
# ------------------------------------------------------------

required_columns = [
    COMPOUND_COL,
    BANDGAP_TARGET,
    HYDROGEN_TARGET,
    DOI_COL
]

missing_columns = [
    c for c in required_columns
    if c not in db_ml.columns
]

if missing_columns:
    raise RuntimeError(
        f"Required columns missing: {missing_columns}"
    )

# ------------------------------------------------------------
# Target summary
# ------------------------------------------------------------

print("Targets and identifiers validated.")

print("\nCompound column:")
print(COMPOUND_COL)

print("\nBandgap target:")
print(BANDGAP_TARGET)

print("\nHydrogen target:")
print(HYDROGEN_TARGET)

print("\nDOI column:")
print(DOI_COL)

print("\nTarget missingness:")

print(
    db_ml[
        [BANDGAP_TARGET, HYDROGEN_TARGET]
    ]
    .isna()
    .sum()
)

print("\nTarget data types:")

print(
    db_ml[
        [BANDGAP_TARGET, HYDROGEN_TARGET]
    ].dtypes
)

In [0]:
# ============================================================
# CMD 03 — TARGET VALUE REPRESENTATION AUDIT
# ============================================================

for target in [
    BANDGAP_TARGET,
    HYDROGEN_TARGET
]:

    print("\n" + "=" * 70)
    print(f"TARGET: {target}")
    print("=" * 70)

    print("\nData type:")
    print(db_ml[target].dtype)

    print("\nNumber of unique values:")
    print(db_ml[target].nunique(dropna=False))

    print("\nTop value representations:")
    display(
        db_ml[target]
        .value_counts(dropna=False)
        .head(20)
        .to_frame("count")
    )

    print("\nEmpty strings:")
    print(
        int(
            db_ml[target]
            .astype(str)
            .str.strip()
            .eq("")
            .sum()
        )
    )

    print("\nLiteral missing strings:")

    missing_tokens = [
        "nan",
        "NaN",
        "None",
        "none",
        "null",
        "NULL",
        "N/A",
        "n/a",
        "NA",
        "na",
        "-"
    ]

    token_counts = {}

    values_as_string = (
        db_ml[target]
        .astype(str)
        .str.strip()
    )

    for token in missing_tokens:
        count = int(
            values_as_string
            .eq(token)
            .sum()
        )

        if count > 0:
            token_counts[token] = count

    print(
        token_counts
        if token_counts
        else "None found."
    )

In [0]:
# ============================================================
# CMD 04 — NUMERIC TARGET CONVERSION AUDIT
# ============================================================

# Create numeric versions without modifying the original columns

db_ml["bandgap_target"] = pd.to_numeric(
    db_ml[BANDGAP_TARGET],
    errors="coerce"
)

db_ml["hydrogen_target"] = pd.to_numeric(
    db_ml[HYDROGEN_TARGET],
    errors="coerce"
)

# ------------------------------------------------------------
# Conversion audit
# ------------------------------------------------------------

print("=" * 70)
print("TARGET CONVERSION AUDIT")
print("=" * 70)

for original, numeric in [
    (BANDGAP_TARGET, "bandgap_target"),
    (HYDROGEN_TARGET, "hydrogen_target")
]:

    print(f"\n{original}")
    print("-" * 70)

    original_non_null = db_ml[original].notna().sum()
    numeric_non_null = db_ml[numeric].notna().sum()
    numeric_missing = db_ml[numeric].isna().sum()

    print("Original non-null:", original_non_null)
    print("Numeric non-null:", numeric_non_null)
    print("Numeric missing:", numeric_missing)

    # Values that failed numeric conversion
    failed_conversion = db_ml.loc[
        db_ml[original].notna() &
        db_ml[numeric].isna(),
        original
    ]

    print("\nValues that failed numeric conversion:")
    
    if len(failed_conversion) > 0:
        display(
            failed_conversion
            .value_counts()
            .to_frame("count")
        )
    else:
        print("None")

# ------------------------------------------------------------
# Final numeric target summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("NUMERIC TARGET SUMMARY")
print("=" * 70)

display(
    db_ml[
        [
            "bandgap_target",
            "hydrogen_target"
        ]
    ].describe()
)

In [0]:
# ============================================================
# CMD 05 — CORRECT LOCALE-BASED TARGET CONVERSION
# ============================================================

# ------------------------------------------------------------
# Bandgap
# ------------------------------------------------------------

db_ml["bandgap_target"] = (
    db_ml[BANDGAP_TARGET]
    .astype(str)
    .str.strip()
    .replace("-", np.nan)
)

db_ml["bandgap_target"] = pd.to_numeric(
    db_ml["bandgap_target"],
    errors="coerce"
)

# ------------------------------------------------------------
# Hydrogen production rate
# ------------------------------------------------------------

db_ml["hydrogen_target"] = (
    db_ml[HYDROGEN_TARGET]
    .astype(str)
    .str.strip()
    .str.replace(",", ".", regex=False)
)

db_ml["hydrogen_target"] = pd.to_numeric(
    db_ml["hydrogen_target"],
    errors="coerce"
)

# ------------------------------------------------------------
# Final conversion audit
# ------------------------------------------------------------

print("=" * 70)
print("FINAL TARGET CONVERSION")
print("=" * 70)

for target in [
    "bandgap_target",
    "hydrogen_target"
]:

    print(f"\n{target}")
    print("-" * 70)

    print("Numeric values:", db_ml[target].notna().sum())
    print("Missing values:", db_ml[target].isna().sum())

    print("Minimum:", db_ml[target].min())
    print("Maximum:", db_ml[target].max())

print("\nTarget dtypes:")

print(
    db_ml[
        [
            "bandgap_target",
            "hydrogen_target"
        ]
    ].dtypes
)

In [0]:
# ============================================================
# CMD 06 — RECONSTRUCT CONSERVATIVE COMPOSITION VALIDITY
# ============================================================

# ------------------------------------------------------------
# Load elemental property table
# ------------------------------------------------------------

atom_props_spark = spark.table(
    "ml_perovskites.atom_props"
)

atom_props_pd = atom_props_spark.toPandas()

print("atom_props shape:", atom_props_pd.shape)

# ------------------------------------------------------------
# Build valid element list
# ------------------------------------------------------------

VALID_ELEMENTS = set(
    atom_props_pd["Element"]
    .dropna()
    .astype(str)
    .str.strip()
)

print("Valid elements in atom_props:", len(VALID_ELEMENTS))

# ------------------------------------------------------------
# Conservative parser
# ------------------------------------------------------------

ELEMENT_REGEX = re.compile(
    r"([A-Z][a-z]?)([0-9]*\.?[0-9]*)"
)

VARIABLE_PATTERN = re.compile(
    r"[xyzXYZ]"
)

def parse_formula_conservative(formula):

    if pd.isna(formula):
        return {}, "missing"

    formula = str(formula).strip()

    if not formula:
        return {}, "missing"

    if VARIABLE_PATTERN.search(formula):
        return {}, "variable_stoichiometry"

    if "/" in formula:
        return {}, "special_notation"

    if any(
        token.lower() in formula.lower()
        for token in ["TBA", "SAC", "Gb"]
    ):
        return {}, "special_notation"

    matches = ELEMENT_REGEX.findall(formula)

    if not matches:
        return {}, "unparsed"

    composition = {}

    for element, amount in matches:

        if element not in VALID_ELEMENTS:
            return {}, "invalid_element"

        if amount == "":
            amount = 1.0
        else:
            amount = float(amount)

        composition[element] = (
            composition.get(element, 0.0)
            + amount
        )

    return composition, "valid"

# ------------------------------------------------------------
# Apply parser
# ------------------------------------------------------------

parsed_results = (
    db_ml[COMPOUND_COL]
    .apply(parse_formula_conservative)
)

db_ml["parsed_composition"] = (
    parsed_results
    .apply(lambda x: x[0])
)

db_ml["parse_status"] = (
    parsed_results
    .apply(lambda x: x[1])
)

db_ml["n_elements"] = (
    db_ml["parsed_composition"]
    .apply(len)
)

# ------------------------------------------------------------
# Audit
# ------------------------------------------------------------

print("\nParse status:")

display(
    db_ml["parse_status"]
    .value_counts()
    .to_frame("count")
)

print("\nTotal rows:", len(db_ml))

print(
    "Valid composition rows:",
    (db_ml["parse_status"] == "valid").sum()
)

print(
    "Invalid/non-modelable composition rows:",
    (db_ml["parse_status"] != "valid").sum()
)

In [0]:
# ============================================================
# CMD 07 — GENERATE ELEMENTAL DESCRIPTORS
# ============================================================

# ------------------------------------------------------------
# Build elemental property lookup
# ------------------------------------------------------------

element_property_columns = [
    c for c in atom_props_pd.columns
    if c != "Element"
]

element_property_lookup = (
    atom_props_pd
    .set_index("Element")
    .to_dict(orient="index")
)

print(
    "Number of elemental properties:",
    len(element_property_columns)
)

# ------------------------------------------------------------
# Weighted descriptor function
# ------------------------------------------------------------

def weighted_descriptor(
    composition,
    property_name
):

    if not composition:
        return np.nan

    values = []
    weights = []

    for element, fraction in composition.items():

        if element not in element_property_lookup:
            continue

        raw_value = (
            element_property_lookup[element]
            .get(property_name)
        )

        value = pd.to_numeric(
            raw_value,
            errors="coerce"
        )

        if pd.isna(value):
            continue

        values.append(float(value))
        weights.append(float(fraction))

    if not values:
        return np.nan

    values = np.array(values)
    weights = np.array(weights)

    weight_sum = weights.sum()

    if weight_sum <= 0:
        return np.nan

    return np.sum(
        values * weights
    ) / weight_sum


# ------------------------------------------------------------
# Generate descriptors
# ------------------------------------------------------------

descriptor_data = {}

for property_name in element_property_columns:

    descriptor_name = f"mean_{property_name}"

    descriptor_data[descriptor_name] = (
        db_ml["parsed_composition"]
        .apply(
            lambda composition:
            weighted_descriptor(
                composition,
                property_name
            )
        )
    )

X_descriptors = pd.DataFrame(
    descriptor_data,
    index=db_ml.index
)

print("\nDescriptor matrix:")
print(X_descriptors.shape)

print(
    "\nExpected:",
    "(440, 58)"
)

In [0]:
# ============================================================
# CMD 08 — APPLY VALIDATED FINAL DESCRIPTOR SET
# ============================================================

final_descriptor_candidates = [
    "mean_atomic_number",
    "mean_atomic_radius",
    "mean_atomic_radius_rahm",
    "mean_atomic_volume",
    "mean_boiling_point",
    "mean_bulk_modulus",
    "mean_covalent_radius_cordero",
    "mean_covalent_radius_pyykko_double",
    "mean_covalent_radius_pyykko_triple",
    "mean_density",
    "mean_dipole_polarizability",
    "mean_electron_negativity",
    "mean_electron_affinity",
    "mean_en_ghosh",
    "mean_first_ion_en",
    "mean_fusion_enthalpy",
    "mean_gs_bandgap",
    "mean_gs_energy",
    "mean_gs_est_bcc_latcnt",
    "mean_gs_mag_moment",
    "mean_hhi_p",
    "mean_hhi_r",
    "mean_heat_capacity_mass",
    "mean_heat_capacity_molar",
    "mean_icsd_volume",
    "mean_heat_of_formation",
    "mean_lattice_constant",
    "mean_mendeleev_number",
    "mean_melting_point",
    "mean_num_unfilled",
    "mean_num_valance",
    "mean_num_d_unfilled",
    "mean_num_d_valence",
    "mean_num_f_valence",
    "mean_num_p_unfilled",
    "mean_num_p_valence",
    "mean_num_s_unfilled",
    "mean_period",
    "mean_specific_heat",
    "mean_thermal_conductivity",
    "mean_vdw_radius_mm3",
    "mean_vdw_radius_uff",
    "mean_sound_velocity"
]

# ------------------------------------------------------------
# Validate feature availability
# ------------------------------------------------------------

missing_features = [
    c for c in final_descriptor_candidates
    if c not in X_descriptors.columns
]

if missing_features:
    raise RuntimeError(
        f"Final descriptors missing from generated matrix: "
        f"{missing_features}"
    )

# ------------------------------------------------------------
# Create reduced descriptor matrix
# ------------------------------------------------------------

X_reduced = X_descriptors[
    final_descriptor_candidates
].copy()

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Final descriptor matrix:")
print(X_reduced.shape)

print("\nNumber of descriptors:")
print(X_reduced.shape[1])

print("\nNumber of rows:")
print(X_reduced.shape[0])

print("\nUnique descriptors:")
print(len(set(final_descriptor_candidates)))

assert X_reduced.shape == (440, 43)
assert len(set(final_descriptor_candidates)) == 43

print("\nValidation passed.")

In [0]:
# ============================================================
# CMD 09 — CONSTRUCT TARGET-SPECIFIC MODELING DATASETS
# ============================================================

# ------------------------------------------------------------
# Base metadata
# ------------------------------------------------------------

metadata = db_ml[
    [
        COMPOUND_COL,
        DOI_COL,
        "parse_status",
        "n_elements"
    ]
].copy()

# ------------------------------------------------------------
# Combine metadata + descriptors + targets
# ------------------------------------------------------------

modeling_data = pd.concat(
    [
        metadata,
        X_reduced,
        db_ml[
            [
                "bandgap_target",
                "hydrogen_target"
            ]
        ]
    ],
    axis=1
)

# ------------------------------------------------------------
# Composition-valid mask
# ------------------------------------------------------------

composition_valid = (
    modeling_data["parse_status"] == "valid"
)

# ------------------------------------------------------------
# Bandgap dataset
# ------------------------------------------------------------

bandgap_data = modeling_data[
    composition_valid &
    modeling_data["bandgap_target"].notna()
].copy()

# ------------------------------------------------------------
# Hydrogen dataset
# ------------------------------------------------------------

hydrogen_data = modeling_data[
    composition_valid &
    modeling_data["hydrogen_target"].notna()
].copy()

# ------------------------------------------------------------
# Validation summary
# ------------------------------------------------------------

print("=" * 70)
print("MODELING DATASET CONSTRUCTION")
print("=" * 70)

print("\nFull modeling data:")
print(modeling_data.shape)

print("\nBandgap dataset:")
print(bandgap_data.shape)

print("\nHydrogen dataset:")
print(hydrogen_data.shape)

print("\nComposition-valid compounds:")
print(composition_valid.sum())

print("\nBandgap usable targets:")
print(bandgap_data["bandgap_target"].notna().sum())

print("\nHydrogen usable targets:")
print(hydrogen_data["hydrogen_target"].notna().sum())

In [0]:
# ============================================================
# CMD 10 — DUPLICATE COMPOUND AUDIT
# ============================================================

for name, dataset in [
    ("Bandgap", bandgap_data),
    ("Hydrogen", hydrogen_data)
]:

    print("\n" + "=" * 70)
    print(f"{name} DATASET — COMPOUND DUPLICATE AUDIT")
    print("=" * 70)

    duplicate_mask = (
        dataset[COMPOUND_COL]
        .duplicated(keep=False)
    )

    duplicate_rows = dataset[
        duplicate_mask
    ].sort_values(COMPOUND_COL)

    print(
        "Rows belonging to duplicated compounds:",
        len(duplicate_rows)
    )

    print(
        "Unique duplicated compounds:",
        duplicate_rows[COMPOUND_COL].nunique()
    )

    if len(duplicate_rows) > 0:

        display(
            duplicate_rows[
                [
                    COMPOUND_COL,
                    DOI_COL,
                    "bandgap_target",
                    "hydrogen_target"
                ]
            ]
        )

    else:
        print("No duplicated compounds found.")

In [0]:
# ============================================================
# CMD 11 — UNIQUE COMPOUND AUDIT
# ============================================================

for name, dataset in [
    ("Bandgap", bandgap_data),
    ("Hydrogen", hydrogen_data)
]:

    print("\n" + "=" * 70)
    print(f"{name} DATASET — UNIQUE COMPOUND SUMMARY")
    print("=" * 70)

    n_rows = len(dataset)

    n_unique = (
        dataset[COMPOUND_COL]
        .nunique()
    )

    n_repeated = (
        dataset[COMPOUND_COL]
        .value_counts()
        .gt(1)
        .sum()
    )

    print("Total experimental rows:", n_rows)
    print("Unique compounds:", n_unique)
    print("Compounds appearing >1 time:", n_repeated)

    print(
        "Rows per unique compound — median:",
        dataset[COMPOUND_COL].value_counts().median()
    )

    print(
        "Rows per unique compound — maximum:",
        dataset[COMPOUND_COL].value_counts().max()
    )

    print("\nFrequency of observations per compound:")

    display(
        dataset[COMPOUND_COL]
        .value_counts()
        .value_counts()
        .sort_index()
        .to_frame("number_of_compounds")
        .rename_axis("observations_per_compound")
    )

In [0]:
# ============================================================
# CMD 12 — REPEATED COMPOUND DESCRIPTOR CONSISTENCY
# ============================================================

for name, dataset in [
    ("Bandgap", bandgap_data),
    ("Hydrogen", hydrogen_data)
]:

    print("\n" + "=" * 70)
    print(f"{name} — REPEATED COMPOUND DESCRIPTOR CONSISTENCY")
    print("=" * 70)

    repeated_compounds = (
        dataset[COMPOUND_COL]
        .value_counts()
        .loc[lambda x: x > 1]
        .index
    )

    repeated_data = dataset[
        dataset[COMPOUND_COL].isin(repeated_compounds)
    ].copy()

    descriptor_variability = (
        repeated_data
        .groupby(COMPOUND_COL)[final_descriptor_candidates]
        .nunique(dropna=False)
    )

    inconsistent_features = (
        descriptor_variability > 1
    ).any(axis=1)

    print(
        "Repeated compounds:",
        len(repeated_compounds)
    )

    print(
        "Repeated compounds with inconsistent descriptors:",
        inconsistent_features.sum()
    )

    if inconsistent_features.sum() > 0:

        display(
            descriptor_variability.loc[
                inconsistent_features
            ]
        )

    else:
        print(
            "All repeated compounds have identical "
            "composition descriptors."
        )

In [0]:
# ============================================================
# CMD 13 — FINAL DATASET INTEGRITY AUDIT
# ============================================================

for name, dataset in [
    ("Bandgap", bandgap_data),
    ("Hydrogen", hydrogen_data)
]:

    print("\n" + "=" * 70)
    print(f"{name} — FINAL DATASET INTEGRITY")
    print("=" * 70)

    # --------------------------------------------------------
    # Descriptor missingness
    # --------------------------------------------------------

    descriptor_missing = (
        dataset[final_descriptor_candidates]
        .isna()
        .sum()
    )

    features_with_missing = (
        descriptor_missing[
            descriptor_missing > 0
        ]
    )

    print(
        "\nFeatures with missing values:",
        len(features_with_missing)
    )

    if len(features_with_missing) > 0:
        display(
            features_with_missing
            .sort_values(ascending=False)
            .to_frame("missing")
        )

    # --------------------------------------------------------
    # Target missingness
    # --------------------------------------------------------

    target_col = (
        "bandgap_target"
        if name == "Bandgap"
        else "hydrogen_target"
    )

    print(
        "\nMissing target values:",
        dataset[target_col].isna().sum()
    )

    # --------------------------------------------------------
    # Compound missingness
    # --------------------------------------------------------

    print(
        "Missing compound identifiers:",
        dataset[COMPOUND_COL].isna().sum()
    )

    # --------------------------------------------------------
    # DOI missingness
    # --------------------------------------------------------

    print(
        "Missing DOI values:",
        dataset[DOI_COL].isna().sum()
    )

    # --------------------------------------------------------
    # Duplicate rows
    # --------------------------------------------------------

    print(
        "Exact duplicate rows:",
        dataset.duplicated().sum()
    )

In [0]:
# ============================================================
# CMD 14 — IDENTIFY EXACT DUPLICATE RECORDS
# ============================================================

for name, dataset in [
    ("Bandgap", bandgap_data),
    ("Hydrogen", hydrogen_data)
]:

    print("\n" + "=" * 70)
    print(f"{name} — EXACT DUPLICATE RECORD AUDIT")
    print("=" * 70)

    duplicate_mask = dataset.duplicated(
        keep=False
    )

    exact_duplicates = (
        dataset[
            duplicate_mask
        ]
        .sort_values(
            by=[
                COMPOUND_COL,
                DOI_COL
            ]
        )
    )

    print(
        "Rows involved in exact duplicates:",
        len(exact_duplicates)
    )

    print(
        "Unique duplicate row patterns:",
        dataset.duplicated().sum()
    )

    display(
        exact_duplicates[
            [
                COMPOUND_COL,
                DOI_COL,
                "bandgap_target",
                "hydrogen_target"
            ]
        ]
    )

In [0]:
# ============================================================
# CMD 15 — REMOVE EXACT DUPLICATE RECORDS
# ============================================================

print("=" * 70)
print("EXACT DUPLICATE REMOVAL")
print("=" * 70)

bandgap_before = len(bandgap_data)
hydrogen_before = len(hydrogen_data)

# Remove only fully identical rows
bandgap_data = (
    bandgap_data
    .drop_duplicates()
    .reset_index(drop=True)
)

hydrogen_data = (
    hydrogen_data
    .drop_duplicates()
    .reset_index(drop=True)
)

print("\nBandgap:")
print("Before:", bandgap_before)
print("After:", len(bandgap_data))
print("Removed:", bandgap_before - len(bandgap_data))

print("\nHydrogen:")
print("Before:", hydrogen_before)
print("After:", len(hydrogen_data))
print("Removed:", hydrogen_before - len(hydrogen_data))

# ------------------------------------------------------------
# Verify no exact duplicates remain
# ------------------------------------------------------------

print("\nRemaining exact duplicates:")

print(
    "Bandgap:",
    bandgap_data.duplicated().sum()
)

print(
    "Hydrogen:",
    hydrogen_data.duplicated().sum()
)

In [0]:
# ============================================================
# CMD 16 — COMPOUND-GROUPED TRAIN / TEST SPLIT
# ============================================================

from sklearn.model_selection import GroupShuffleSplit

TEST_SIZE = 0.20
RANDOM_STATE = 42

def grouped_train_test_split(
    dataset,
    group_column,
    test_size=0.20,
    random_state=42
):

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=test_size,
        random_state=random_state
    )

    train_idx, test_idx = next(
        splitter.split(
            dataset,
            groups=dataset[group_column]
        )
    )

    train = (
        dataset
        .iloc[train_idx]
        .copy()
        .reset_index(drop=True)
    )

    test = (
        dataset
        .iloc[test_idx]
        .copy()
        .reset_index(drop=True)
    )

    return train, test


# ------------------------------------------------------------
# Bandgap split
# ------------------------------------------------------------

bandgap_train, bandgap_test = grouped_train_test_split(
    bandgap_data,
    COMPOUND_COL,
    TEST_SIZE,
    RANDOM_STATE
)

# ------------------------------------------------------------
# Hydrogen split
# ------------------------------------------------------------

hydrogen_train, hydrogen_test = grouped_train_test_split(
    hydrogen_data,
    COMPOUND_COL,
    TEST_SIZE,
    RANDOM_STATE
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

for name, train, test in [
    ("Bandgap", bandgap_train, bandgap_test),
    ("Hydrogen", hydrogen_train, hydrogen_test)
]:

    print("\n" + "=" * 70)
    print(f"{name} — GROUPED TRAIN / TEST SPLIT")
    print("=" * 70)

    print("Train rows:", len(train))
    print("Test rows:", len(test))

    print(
        "Train unique compounds:",
        train[COMPOUND_COL].nunique()
    )

    print(
        "Test unique compounds:",
        test[COMPOUND_COL].nunique()
    )

    print(
        "Train percentage:",
        round(100 * len(train) / (len(train) + len(test)), 2)
    )

    print(
        "Test percentage:",
        round(100 * len(test) / (len(train) + len(test)), 2)
    )

In [0]:
# ============================================================
# CMD 17 — VERIFY ZERO COMPOUND OVERLAP
# ============================================================

for name, train, test in [
    ("Bandgap", bandgap_train, bandgap_test),
    ("Hydrogen", hydrogen_train, hydrogen_test)
]:

    train_compounds = set(
        train[COMPOUND_COL]
    )

    test_compounds = set(
        test[COMPOUND_COL]
    )

    overlap = (
        train_compounds
        .intersection(test_compounds)
    )

    print("\n" + "=" * 70)
    print(f"{name} — GROUP LEAKAGE CHECK")
    print("=" * 70)

    print(
        "Train compounds:",
        len(train_compounds)
    )

    print(
        "Test compounds:",
        len(test_compounds)
    )

    print(
        "Overlapping compounds:",
        len(overlap)
    )

    if overlap:
        print("\nOVERLAPPING COMPOUNDS:")
        print(sorted(overlap))

        raise RuntimeError(
            f"Group leakage detected in {name} dataset."
        )

    print("PASS — No compound appears in both train and test.")

In [0]:
# ============================================================
# CMD 18 — FINAL MODELING MATRICES
# ============================================================

# ------------------------------------------------------------
# Feature matrices
# ------------------------------------------------------------

X_bandgap_train = bandgap_train[
    final_descriptor_candidates
].copy()

X_bandgap_test = bandgap_test[
    final_descriptor_candidates
].copy()

X_hydrogen_train = hydrogen_train[
    final_descriptor_candidates
].copy()

X_hydrogen_test = hydrogen_test[
    final_descriptor_candidates
].copy()

# ------------------------------------------------------------
# Target vectors
# ------------------------------------------------------------

y_bandgap_train = bandgap_train[
    "bandgap_target"
].copy()

y_bandgap_test = bandgap_test[
    "bandgap_target"
].copy()

y_hydrogen_train = hydrogen_train[
    "hydrogen_target"
].copy()

y_hydrogen_test = hydrogen_test[
    "hydrogen_target"
].copy()

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

print("=" * 70)
print("FINAL MODELING MATRICES")
print("=" * 70)

print("\nBANDGAP")
print("X_train:", X_bandgap_train.shape)
print("X_test: ", X_bandgap_test.shape)
print("y_train:", y_bandgap_train.shape)
print("y_test: ", y_bandgap_test.shape)

print("\nHYDROGEN")
print("X_train:", X_hydrogen_train.shape)
print("X_test: ", X_hydrogen_test.shape)
print("y_train:", y_hydrogen_train.shape)
print("y_test: ", y_hydrogen_test.shape)

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert X_bandgap_train.shape[1] == 43
assert X_bandgap_test.shape[1] == 43

assert X_hydrogen_train.shape[1] == 43
assert X_hydrogen_test.shape[1] == 43

assert len(X_bandgap_train) == len(y_bandgap_train)
assert len(X_bandgap_test) == len(y_bandgap_test)

assert len(X_hydrogen_train) == len(y_hydrogen_train)
assert len(X_hydrogen_test) == len(y_hydrogen_test)

assert X_bandgap_train.isna().sum().sum() == 0
assert X_bandgap_test.isna().sum().sum() == 0

assert X_hydrogen_train.isna().sum().sum() == 0
assert X_hydrogen_test.isna().sum().sum() == 0

assert y_bandgap_train.isna().sum() == 0
assert y_bandgap_test.isna().sum() == 0

assert y_hydrogen_train.isna().sum() == 0
assert y_hydrogen_test.isna().sum() == 0

print("\nAll final matrix validations passed.")

In [0]:
# ============================================================
# CMD 19 — EXPORT FINAL MODELING DATASETS
# ============================================================

# ------------------------------------------------------------
# Output columns
# ------------------------------------------------------------

bandgap_output = bandgap_train[
    [
        COMPOUND_COL,
        DOI_COL
    ] + final_descriptor_candidates + [
        "bandgap_target"
    ]
].copy()

bandgap_test_output = bandgap_test[
    [
        COMPOUND_COL,
        DOI_COL
    ] + final_descriptor_candidates + [
        "bandgap_target"
    ]
].copy()

hydrogen_output = hydrogen_train[
    [
        COMPOUND_COL,
        DOI_COL
    ] + final_descriptor_candidates + [
        "hydrogen_target"
    ]
].copy()

hydrogen_test_output = hydrogen_test[
    [
        COMPOUND_COL,
        DOI_COL
    ] + final_descriptor_candidates + [
        "hydrogen_target"
    ]
].copy()

# ------------------------------------------------------------
# Convert to Spark DataFrames
# ------------------------------------------------------------

spark_bandgap_train = spark.createDataFrame(
    bandgap_output
)

spark_bandgap_test = spark.createDataFrame(
    bandgap_test_output
)

spark_hydrogen_train = spark.createDataFrame(
    hydrogen_output
)

spark_hydrogen_test = spark.createDataFrame(
    hydrogen_test_output
)

# ------------------------------------------------------------
# Save as Delta tables
# ------------------------------------------------------------

spark_bandgap_train.write.mode("overwrite").format("delta").saveAsTable(
    "ml_perovskites.ml_bandgap_train"
)

spark_bandgap_test.write.mode("overwrite").format("delta").saveAsTable(
    "ml_perovskites.ml_bandgap_test"
)

spark_hydrogen_train.write.mode("overwrite").format("delta").saveAsTable(
    "ml_perovskites.ml_hydrogen_train"
)

spark_hydrogen_test.write.mode("overwrite").format("delta").saveAsTable(
    "ml_perovskites.ml_hydrogen_test"
)

print("=" * 70)
print("NOTEBOOK 02 EXPORT COMPLETE")
print("=" * 70)

print("\nCreated tables:")

print("ml_perovskites.ml_bandgap_train")
print("ml_perovskites.ml_bandgap_test")
print("ml_perovskites.ml_hydrogen_train")
print("ml_perovskites.ml_hydrogen_test")

Decisions are locked:

- Conservative formula parsing
- 43 composition descriptors
- No descriptor missingness in modeling sets
- Exact duplicate records removed
- Repeated compounds retained as experimental observations
- Compound-grouped train/test split
- 0 compound overlap between train and test
- Test set has not been used for feature selection, imputation, scaling, or model tuning